# Experimento de Prueba — LightGBM Multivariado + Two-Stage
Objetivo: verificar que el pipeline completo funciona de extremo a extremo.

**Flujo:**
1. Setup
2. Cargar y preparar datos
3. Verificar features
4. Visualizar CV splits
5. Cross-validation — modelo único (baseline)
6. Loggear en MLflow
7. Evaluar en test set
8. Two-stage — baseline end-to-end
9. Análisis del umbral
10. Comparativa: two-stage vs modelo único


## 1. Setup

In [0]:
%pip install lightgbm --quiet
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/carlos.saquel@gmail.com/santiago-weather-forecast')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.data.preprocessing import build_features, train_test_split, get_feature_names
from src.models.lightgbm_model import LightGBMPredictor
from src.models.two_stage_model import TwoStagePredictor
from src.evaluation.metrics import evaluate_model
from src.evaluation.cross_validation import TimeSeriesSplit, run_cv
from src.evaluation.two_stage_cv import run_cv_two_stage, evaluate_two_stage, find_best_threshold
from src.utils.mlflow_utils import setup_experiment, log_cv_run, log_test_evaluation
from src.utils.config import *

setup_experiment()
print('✅ Setup completo')


## 2. Cargar y preparar datos

In [0]:
# Cargar datos crudos desde Delta
df_raw = load_from_delta_table('weather_raw', spark)

# Pipeline completo de features
df = build_features(df_raw)

# Split train / test
df_train, df_test = train_test_split(df)

print(f'\n📋 Columnas totales: {df.shape[1]}')
print(f'   Features X:      {len(get_feature_names(df))}')
print(f'   Target:          {TARGET}')

## 3. Verificar features

In [0]:
feature_names = get_feature_names(df)

print(f'Features ({len(feature_names)} total):')
for i, f in enumerate(feature_names):
    print(f'  {i+1:3d}. {f}')

print(f'\n🔍 NaN por columna en df_train:')
nan_counts = df_train[feature_names + [TARGET]].isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) == 0:
    print('  ✅ Sin NaN')
else:
    print(nan_counts)

print(f'\n📊 Estadísticas del target ({TARGET}):')
print(df_train[TARGET].describe().round(3))

## 4. Visualizar CV splits

In [0]:
cv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding'
)
cv.visualize(df_train)

## 5. Cross-validation

In [0]:
# Parámetros base — punto de partida limpio
# No tocar hasta confirmar que el pipeline funciona end-to-end
BASE_PARAMS = {
    'objective':              'tweedie',
    'tweedie_variance_power': 1.5,
    'n_estimators':           300,
    'learning_rate':          0.05,
    'max_depth':              6,
    'num_leaves':             31,
    'min_child_samples':      20,
    'subsample':              0.8,
    'colsample_bytree':       0.8,
    'reg_alpha':              0.1,
    'reg_lambda':             1.0,
}

print('📋 Parámetros base:')
for k, v in BASE_PARAMS.items():
    print(f'  {k:30s}: {v}')

In [0]:
results_cv = run_cv(
    model_class=LightGBMPredictor,
    df=df_train,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding',
    **BASE_PARAMS
)

display(results_cv)

## 6. Loggear en MLflow

In [0]:
# Con sliding: entrenar con los datos del último fold
cv_temp = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding'
)
last_train, _ = cv_temp.split(df_train)[-1]

model_final = LightGBMPredictor(**BASE_PARAMS)
model_final.fit(last_train)

print(f'📅 Modelo final entrenado con: '
      f'{last_train.index.min().date()} → {last_train.index.max().date()} '
      f'({len(last_train)} días)')

model_final.print_summary()

# Loggear en MLflow
run_id = log_cv_run(
    model=model_final,
    results_df=results_cv,
    model_params=BASE_PARAMS,
    run_name='LightGBM_baseline_v1',
    description='Baseline multivariado — todas las features, parámetros base',
    register_model=False,  # True solo cuando el modelo sea candidato a producción
)

print(f'\n🔗 Run ID: {run_id}')

## 7. Evaluación en test set

In [0]:
import matplotlib.pyplot as plt

# Predicciones en test
preds_test = model_final.predict(df_test)
y_test     = df_test[TARGET]

# Métricas
test_metrics = evaluate_model(y_test, preds_test, model_name='LightGBM_baseline_v1')

# Loggear métricas de test al run de CV
log_test_evaluation(
    run_id=run_id,
    test_metrics=test_metrics,
    description='Evaluación en test set holdout'
)

# Visualización
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('LightGBM Baseline — Evaluación en Test Set', fontsize=14, fontweight='bold')

# Panel 1: serie completa
axes[0].plot(y_test.index, y_test.values,
             label='Real', color='steelblue', alpha=0.8, linewidth=1.5)
axes[0].plot(preds_test.index, preds_test.values,
             label='Predicción', color='#e63946', alpha=0.8, linewidth=1.5)
axes[0].set_title('Precipitación real vs predicha — Test set completo')
axes[0].set_ylabel('mm')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: zoom en meses de invierno (donde hay más señal)
mask_invierno = (y_test.index.month >= 5) & (y_test.index.month <= 9)
axes[1].plot(y_test[mask_invierno].index, y_test[mask_invierno].values,
             label='Real', color='steelblue', alpha=0.8, linewidth=1.5)
axes[1].plot(preds_test[mask_invierno].index, preds_test[mask_invierno].values,
             label='Predicción', color='#e63946', alpha=0.8, linewidth=1.5)
axes[1].set_title('Zoom: Mayo–Septiembre (temporada de lluvia)')
axes[1].set_ylabel('mm')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
importance = model_final.get_feature_importance(top_n=20)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    importance['feature'][::-1],
    importance['importance_pct'][::-1],
    color='steelblue', edgecolor='white', alpha=0.85
)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Importancia (%)')
ax.set_title('Top 20 Features — LightGBM Baseline', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

display(importance)

## 8. Grid Search — Clasificador

Busca la mejor configuración del clasificador (etapa 1).
Se evalúa AUC, F1, Precision y Recall por CV.
El umbral óptimo se busca en validación interna de cada fold (sin tocar test).

In [0]:
# Grid reducido para verificar el pipeline — en 04_grid_search va el grid completo
CLF_GRID = [
    {
        'name':              'clf_base',
        'n_estimators':      200,
        'learning_rate':     0.05,
        'max_depth':         6,
        'num_leaves':        31,
        'min_child_samples': 20,
        'subsample':         0.8,
        'colsample_bytree':  0.8,
    },
    {
        'name':              'clf_scale_pos',
        'n_estimators':      200,
        'learning_rate':     0.05,
        'max_depth':         6,
        'num_leaves':        31,
        'min_child_samples': 20,
        'is_unbalance':      False,        # reemplazar is_unbalance por scale_pos_weight
        'scale_pos_weight':  8.0,          # ~89/11 ratio de clases
        'subsample':         0.8,
        'colsample_bytree':  0.8,
    },
    {
        'name':              'clf_dart',
        'boosting_type':     'dart',
        'n_estimators':      200,
        'learning_rate':     0.1,
        'num_leaves':        40,
        'drop_rate':         0.15,
    },
]

print(f'📋 Configuraciones clf a probar: {len(CLF_GRID)}')
for c in CLF_GRID:
    print(f'  - {c["name"]}')


In [0]:
from src.evaluation.two_stage_cv import run_cv_clf_grid

results_clf_grid = run_cv_clf_grid(
    clf_grid=CLF_GRID,
    df=df_train,
    thresholds=CLF_THRESHOLDS,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    log_mlflow=True,
)

display(results_clf_grid[['name', 'best_threshold', 'avg_auc', 'avg_f1', 'std_f1',
                            'avg_precision', 'avg_recall']])


In [0]:
# Extraer mejor configuración clf
best_clf_row = results_clf_grid.iloc[0]
BEST_CLF_NAME      = best_clf_row['name']
BEST_CLF_THRESHOLD = float(best_clf_row['best_threshold'])

# Reconstruir params del clf ganador (sin columnas extra)
best_cfg = next(c for c in CLF_GRID if c['name'] == BEST_CLF_NAME)
BEST_CLF_PARAMS = {k: v for k, v in best_cfg.items() if k != 'name'}

print(f'🏆 Mejor clasificador: {BEST_CLF_NAME}')
print(f'   Umbral:    {BEST_CLF_THRESHOLD}')
print(f'   avg_F1:    {best_clf_row["avg_f1"]:.3f}')
print(f'   avg_AUC:   {best_clf_row["avg_auc"]:.3f}')
print(f'   avg_Prec:  {best_clf_row["avg_precision"]:.3f}')
print(f'   avg_Rec:   {best_clf_row["avg_recall"]:.3f}')


## 9. Grid Search — Regresor

Fijando el mejor clasificador, busca la mejor configuración del regresor (etapa 2).
Se evalúa RMSE, R² y Recall de picos en días lluviosos reales.

In [0]:
# Grid reducido para verificar el pipeline
REG_GRID = [
    {
        'name':                   'reg_tweedie',
        'objective':              'tweedie',
        'tweedie_variance_power': 1.5,
        'n_estimators':           200,
        'learning_rate':          0.05,
        'max_depth':              6,
        'num_leaves':             31,
        'min_child_samples':      10,
        'subsample':              0.8,
        'colsample_bytree':       0.8,
        'reg_alpha':              0.1,
        'reg_lambda':             1.0,
    },
    {
        'name':              'reg_l1',
        'objective':         'regression_l1',
        'n_estimators':      200,
        'learning_rate':     0.05,
        'max_depth':         6,
        'num_leaves':        31,
        'min_child_samples': 10,
        'subsample':         0.8,
        'colsample_bytree':  0.8,
    },
    {
        'name':                   'reg_tweedie_log',   # igual que tweedie pero con log_target=True
        'objective':              'tweedie',
        'tweedie_variance_power': 1.5,
        'n_estimators':           200,
        'learning_rate':          0.05,
        'max_depth':              6,
        'num_leaves':             31,
        'min_child_samples':      10,
    },
]

print(f'📋 Configuraciones reg a probar: {len(REG_GRID)}')
for r in REG_GRID:
    print(f'  - {r["name"]}')
print(f'\n  Clf fijo:  {BEST_CLF_NAME}')
print(f'  Umbral:    {BEST_CLF_THRESHOLD}')


In [0]:
from src.evaluation.two_stage_cv import run_cv_reg_grid

results_reg_grid = run_cv_reg_grid(
    reg_grid=REG_GRID,
    best_clf_params=BEST_CLF_PARAMS,
    best_threshold=BEST_CLF_THRESHOLD,
    df=df_train,
    log_target=True,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    log_mlflow=True,
)

display(results_reg_grid[['name', 'avg_mae_rain', 'avg_rmse_rain', 'std_rmse_rain',
                            'avg_r2_rain', 'avg_recall_picos', 'avg_rmse']])


In [0]:
# Extraer mejor configuración reg
best_reg_row = results_reg_grid.iloc[0]
BEST_REG_NAME = best_reg_row['name']

best_reg_cfg  = next(r for r in REG_GRID if r['name'] == BEST_REG_NAME)
BEST_REG_PARAMS = {k: v for k, v in best_reg_cfg.items() if k != 'name'}

print(f'🏆 Mejor regresor: {BEST_REG_NAME}')
print(f'   avg_RMSE_rain:    {best_reg_row["avg_rmse_rain"]:.2f} mm')
print(f'   avg_R²_rain:      {best_reg_row["avg_r2_rain"]:.3f}')
print(f'   avg_Recall_picos: {best_reg_row["avg_recall_picos"]*100:.0f}%')


## 10. Evaluación final — Two-Stage vs Modelo único

Reentrenar la combinación ganadora con `df_train` completo y evaluar en test set.
Comparar contra el baseline `regression_l1`.

In [0]:
# ── Modelo único baseline ─────────────────────────────────────────
BEST_SINGLE_PARAMS = {
    'objective':         'regression_l1',
    'n_estimators':      300,
    'learning_rate':     0.05,
    'max_depth':         6,
    'num_leaves':        31,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
}
single_model = LightGBMPredictor(**BEST_SINGLE_PARAMS)
single_model.fit(df_train)
preds_single = single_model.predict(df_test)

# ── Two-stage ganador ─────────────────────────────────────────────
ts_final = TwoStagePredictor(
    clf_params=BEST_CLF_PARAMS,
    reg_params=BEST_REG_PARAMS,
    threshold=BEST_CLF_THRESHOLD,
    log_target=True,
)
ts_final.fit(df_train)
preds_ts = ts_final.predict(df_test)
prob_ts  = ts_final.predict_proba(df_test)

y_test_vals = df_test[TARGET]
print(f'✅ Ambos modelos entrenados con {len(df_train)} días')
print(f'   Test: {len(df_test)} días ({y_test_vals.index.min().date()} → {y_test_vals.index.max().date()})')


In [0]:
from src.evaluation.metrics import evaluate_model
from src.evaluation.two_stage_cv import evaluate_two_stage

metrics_single = evaluate_model(y_test_vals, preds_single, model_name='regression_l1')
metrics_ts     = evaluate_two_stage(
    y_true=y_test_vals, y_pred=preds_ts, prob_llueve=prob_ts,
    threshold=BEST_CLF_THRESHOLD, label='TwoStage',
)

print('\n' + '='*65)
print('COMPARATIVA FINAL — Test Set')
print('='*65)
print(f'  {"Métrica":22s}  {"Modelo único (L1)":>18}  {"Two-Stage":>12}  {"Δ":>8}')
print(f'  {"-"*64}')

comparaciones = [
    ('rmse',         'RMSE (todos)',        True),
    ('r2',           'R² (todos)',          False),
    ('f1_score',     'F1 (clasificación)',  False),
    ('recall',       'Recall lluvia',       False),
    ('recall_picos', 'Recall picos >20mm',  False),
    ('mae_rain',     'MAE (días lluvia)',    True),
    ('rmse_rain',    'RMSE (días lluvia)',   True),
    ('r2_rain',      'R² (días lluvia)',     False),
    ('auc',          'AUC-ROC',             False),
]
for key, label, lower_better in comparaciones:
    v_s  = metrics_single.get(key, np.nan)
    v_ts = metrics_ts.get(key, np.nan)
    if pd.isna(v_s) or pd.isna(v_ts):
        print(f'  {label:22s}  {"—":>18}  {"—":>12}')
        continue
    delta = v_ts - v_s
    mejor = '✅' if (delta < 0 and lower_better) or (delta > 0 and not lower_better) else ('🔴' if delta != 0 else '⚪')
    print(f'  {label:22s}  {v_s:>18.3f}  {v_ts:>12.3f}  {mejor} {delta:+.3f}')


In [0]:
# Visualización en temporada de lluvia (Mayo–Septiembre)
mask_inv = (y_test_vals.index.month >= 5) & (y_test_vals.index.month <= 9)
y_inv    = y_test_vals[mask_inv]

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Comparativa Test Set — Temporada de Lluvia (May–Sep)',
             fontsize=14, fontweight='bold')

for ax, (preds, label, color) in zip(axes, [
    (preds_single[mask_inv], 'Modelo único (regression_l1)', '#e63946'),
    (preds_ts[mask_inv],     f'Two-Stage (thr={BEST_CLF_THRESHOLD:.2f})', '#52b788'),
]):
    ax.fill_between(y_inv.index, 0, y_inv.values, alpha=0.2, color='steelblue', label='Real')
    ax.plot(y_inv.index, y_inv.values, color='steelblue', linewidth=1.2, alpha=0.8)
    ax.plot(preds.index, preds.values, color=color, linewidth=1.5, alpha=0.9, label=label)
    mask_p = y_inv > 20
    ax.scatter(y_inv.index[mask_p], y_inv.values[mask_p],
               color='navy', s=40, zorder=5, label='Picos reales (>20mm)')
    ax.scatter(preds.index[mask_p], preds.values[mask_p],
               color=color, s=40, marker='x', zorder=5, linewidth=2,
               label='Predicción en picos')
    ax.set_ylabel('mm')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [0]:
# Importancia de features — clf vs reg
imp = ts_final.get_feature_importance(top_n=15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Importancia de Features — TwoStage ({BEST_CLF_NAME} + {BEST_REG_NAME})',
             fontsize=13, fontweight='bold')

for ax, (stage, title) in zip(axes, [
    ('clf', '🎯 Clasificador — ¿Llueve mañana?'),
    ('reg', '📏 Regresor — ¿Cuánto llueve?'),
]):
    df_imp = imp[stage]
    bars = ax.barh(df_imp['feature'][::-1], df_imp['importance_pct'][::-1],
                   color='steelblue', edgecolor='white', alpha=0.85)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
    ax.set_xlabel('Importancia (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('\n💡 Si clf y reg priorizan features distintas → señales complementarias correctamente capturadas.')
